In [1]:
library(readr)
library(dplyr)
library(ggplot2)
library(tidyr)
library(stringr)
library(purrr)

Warning message:
“package ‘readr’ was built under R version 4.2.3”

Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message:
“package ‘ggplot2’ was built under R version 4.2.3”
Warning message:
“package ‘tidyr’ was built under R version 4.2.3”
Warning message:
“package ‘stringr’ was built under R version 4.2.3”
Warning message:
“package ‘purrr’ was built under R version 4.2.3”


In [2]:
boot_res <- read_csv("./results/boot_res_plate1_u2os_nest_confluence.csv", show_col_types = FALSE)

dim(boot_res)
names(boot_res)
head(boot_res)

[1] 12600    14

[1] "metric_name"        "cell_line"          "ablation_type"     
 [4] "boot_idx"           "beta_x1_restricted" "beta_x1_full"      
 [7] "beta_x2"            "r2_restricted"      "r2_full"           
[10] "delta_r2"           "partial_r2_x2"      "cohen_f2_x2"       
[13] "n_boot_rows"        "n_group_rows"

metric_name,cell_line,ablation_type,boot_idx,beta_x1_restricted,beta_x1_full,beta_x2,r2_restricted,r2_full,delta_r2,partial_r2_x2,cohen_f2_x2,n_boot_rows,n_group_rows
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
mae,U2-OS,GridDistortion,0,0.01186692,0.01178103,1.445430e-06,0.01661104,0.05673481,0.04012377,0.04080153,0.04253711,7236,14472
mae,U2-OS,GridDistortion,1,0.01335311,0.01350387,1.489711e-06,0.02137225,0.06417254,0.04280029,0.04373500,0.04573523,7236,14472
mae,U2-OS,GridDistortion,2,0.01482024,0.01466528,1.530657e-06,0.02518480,0.06904199,0.04385719,0.04499026,0.04710974,7236,14472
mae,U2-OS,GridDistortion,3,0.01412130,0.01411580,1.518279e-06,0.02327078,0.06738554,0.04411476,0.04516580,0.04730224,7236,14472
mae,U2-OS,GridDistortion,4,0.01411974,0.01397804,1.640792e-06,0.02287725,0.07355346,0.05067621,0.05186269,0.05469955,7236,14472
mae,U2-OS,GridDistortion,5,0.01487855,0.01521756,1.584700e-06,0.02488949,0.07196877,0.04707928,0.04828097,0.05073028,7236,14472


In [3]:
summary(select(boot_res, partial_r2_x2, r2_restricted))

 partial_r2_x2      r2_restricted     
 Min.   :0.000000   Min.   :0.002743  
 1st Qu.:0.002494   1st Qu.:0.044056  
 Median :0.035256   Median :0.131095  
 Mean   :0.052389   Mean   :0.182754  
 3rd Qu.:0.087516   3rd Qu.:0.172235  
 Max.   :0.229738   Max.   :0.906013  

In [4]:
# unique ablation types
ablation_types <- unique(boot_res$ablation_type)
print(ablation_types)

[1] "GridDistortion" "GaussNoise"     "GaussianBlur"   "Erode"         
[5] "Dilate"         "RandomGamma"   


In [5]:
plot_partial_r2_vs_r2 <- function(
  boot_res,
  panel_cols,
  hue_col = "metric_name",
  partial_col = "partial_r2_x2",
  r2_col = "r2_restricted",
  partial_label = NULL,
  r2_label = NULL,
  n_cols = 3,
  title_wrap_width = 48,
  metric_palette = NULL,
  metric_order = NULL,
  band_fills = c(
    "<= 10% of restricted variance explained" = "#d9f0d3",
    "10-50% of restricted variance explained" = "#fee08b",
    "50-100% of restricted variance explained" = "#fcbba1"
  ),
  shade_alpha = 0.22,
  show_band_legend = TRUE,
  free_scales = "fixed"
) {
  # --------------------------
  # 1. Aggregate mean and 95% CI
  # --------------------------
  group_cols <- c(panel_cols, hue_col)

  grouped_stats <- boot_res %>%
    group_by(across(all_of(group_cols))) %>%
    summarise(
      partial_r2_mean  = mean(.data[[partial_col]], na.rm = TRUE),
      partial_r2_lower = quantile(.data[[partial_col]], 0.025, na.rm = TRUE),
      partial_r2_upper = quantile(.data[[partial_col]], 0.975, na.rm = TRUE),
      r2_restricted_mean  = mean(.data[[r2_col]], na.rm = TRUE),
      r2_restricted_lower = quantile(.data[[r2_col]], 0.025, na.rm = TRUE),
      r2_restricted_upper = quantile(.data[[r2_col]], 0.975, na.rm = TRUE),
      .groups = "drop"
    )

  # --------------------------
  # 2. Build facet labels from panel columns
  # --------------------------
  grouped_stats <- grouped_stats %>%
    mutate(
      panel_label = pmap_chr(
        select(., all_of(panel_cols)),
        function(...) {
          vals <- list(...)
          txt <- paste(
            purrr::map2_chr(panel_cols, vals, ~ paste0(.x, "=", .y)),
            collapse = " | "
          )
          str_wrap(txt, width = title_wrap_width)
        }
      )
    )

  if (!is.null(metric_order)) {
    grouped_stats[[hue_col]] <- factor(
      grouped_stats[[hue_col]],
      levels = metric_order
    )
  }

  # --------------------------
  # 3. Build shaded threshold regions only
  # partial R² = (c * x) / (1 - x)
  # --------------------------
  x_all <- seq(0.001, 0.999, length.out = 800)

  curve_fun <- function(r, x) {
    y <- (r * x) / (1 - x)
    x_max <- 1 / (1 + r)
    y[x >= x_max] <- NA_real_
    y
  }

  band_df <- tibble::tibble(
    x = x_all,
    y10 = curve_fun(0.1, x_all),
    y50 = curve_fun(0.5, x_all),
    y100 = curve_fun(1.0, x_all)
  )

  band_low <- band_df %>%
    transmute(
      x = x,
      ymin = 0,
      ymax = y10,
      band = "<= 10% of restricted variance explained"
    ) %>%
    filter(!is.na(ymax))

  band_mid <- band_df %>%
    transmute(
      x = x,
      ymin = y10,
      ymax = y50,
      band = "10-50% of restricted variance explained"
    ) %>%
    filter(!is.na(ymin), !is.na(ymax))

  band_high <- band_df %>%
    transmute(
      x = x,
      ymin = y50,
      ymax = y100,
      band = "50-100% of restricted variance explained"
    ) %>%
    filter(!is.na(ymin), !is.na(ymax))

  band_plot_df <- bind_rows(band_low, band_mid, band_high) %>%
    mutate(
      band = factor(
        band,
        levels = c(
          "<= 10% of restricted variance explained",
          "10-50% of restricted variance explained",
          "50-100% of restricted variance explained"
        )
      )
    )

  # --------------------------
  # 4. Default axis labels
  # --------------------------
  if (is.null(partial_label)) {
    partial_label <- paste0("Partial R² (", partial_col, ")")
  }
  if (is.null(r2_label)) {
    r2_label <- paste0("Restricted R² (", r2_col, ")")
  }

  # --------------------------
  # 5. Validate free_scales
  # --------------------------
  allowed_scales <- c("fixed", "free", "free_x", "free_y")
  if (!(free_scales %in% allowed_scales)) {
    stop("free_scales must be one of: ", paste(allowed_scales, collapse = ", "))
  }

  # --------------------------
  # 6. Base plot
  # --------------------------
  p <- ggplot(
    grouped_stats,
    aes(
      x = r2_restricted_mean,
      y = partial_r2_mean,
      color = .data[[hue_col]]
    )
  ) +
    geom_ribbon(
      data = band_plot_df,
      aes(x = x, ymin = ymin, ymax = ymax, fill = band),
      inherit.aes = FALSE,
      alpha = shade_alpha,
      color = NA
    ) +
    geom_errorbar(
      aes(
        ymin = partial_r2_lower,
        ymax = partial_r2_upper
      ),
      width = 0,
      alpha = 0.7,
      linewidth = 0.4
    ) +
    geom_segment(
      aes(
        x = r2_restricted_lower,
        xend = r2_restricted_upper,
        y = partial_r2_mean,
        yend = partial_r2_mean
      ),
      alpha = 0.7,
      linewidth = 0.4
    ) +
    geom_point(size = 2.4, alpha = 0.95) +
    facet_wrap(~ panel_label, ncol = n_cols, scales = free_scales) +
    labs(
      x = r2_label,
      y = partial_label,
      color = NULL,
      fill = NULL
    ) +
    coord_cartesian(
      xlim = c(0, NA),
      ylim = c(0, NA),
      expand = TRUE
    ) +
    theme_bw(base_size = 11) +
    theme(
      panel.grid.minor = element_blank(),
      panel.grid.major = element_line(color = "grey90", linewidth = 0.3),
      strip.text = element_text(size = 10, face = "bold"),
      legend.position = "right"
    )

  # --------------------------
  # 7. Metric color palette
  # --------------------------
  if (!is.null(metric_order)) {
    hue_levels <- metric_order
  } else {
    hue_levels <- sort(unique(as.character(grouped_stats[[hue_col]])))
  }

  if (!is.null(metric_palette)) {
    if (is.null(names(metric_palette))) {
      if (length(metric_palette) < length(hue_levels)) {
        stop("Unnamed metric_palette must have at least as many colors as hue levels.")
      }
      names(metric_palette) <- hue_levels
    }
    p <- p + scale_color_manual(
      values = metric_palette,
      breaks = hue_levels
    )
  } else {
    default_metric_cols <- scales::hue_pal()(length(hue_levels))
    names(default_metric_cols) <- hue_levels
    p <- p + scale_color_manual(
      values = default_metric_cols,
      breaks = hue_levels
    )
  }

  p <- p + scale_fill_manual(
    values = band_fills,
    drop = FALSE
  )

  if (!show_band_legend) {
    p <- p + guides(fill = "none")
  }

  return(p)
}

In [6]:
metric_pal <- c(
  "lpips"   = "#5E3C99",  # deep purple
  "dists"   = "#1F78B4",  # strong blue

  "ssim"    = "#1B9E77",  # green
  "foreground_ssim" = "#66C2A5",  # lighter related green

  "psnr"    = "#E66101",  # orange
  "foreground_psnr" = "#FDB863",   # lighter related orange

  "mae" = "#CCCCCC"  # gray
)

for (ablation_type in ablation_types) {
  
  p <- plot_partial_r2_vs_r2(
    boot_res = boot_res %>% filter(ablation_type == !!ablation_type),
    panel_cols = c("cell_line", "ablation_type"),
    hue_col = "metric_name",
    partial_col = "partial_r2_x2",
    r2_col = "r2_restricted",
    partial_label = "Partial R²",
    r2_label = "Restricted R²",
    n_cols = 1,
    shade_alpha = 0.4,
    free_scales = "free",
    metric_palette = metric_pal,
    metric_order = c("dists", "lpips", "foreground_ssim", "ssim", "foreground_psnr", "psnr", "mae")
  )

  ggsave(
    paste0("./plots/plate2_u2os_nest_confluence_", ablation_type, ".pdf"),
    p,
    width = 7.5,
    height = 4.5,
  )

}